# Path-Norm Quickstart

This notebook shows how to use the public functions exposed by `pathnorm.py` with standard PyTorch / torchvision models.

Call `compute_path_norm(...)` or `compute_path_norms(...)` directly on a model. If the model is unsupported, the computation raises a precise error pointing to the offending module or operation.

Supported models include modern ReLU networks with skip connections, max-pooling, average-pooling, convolutions, and inference-time batch normalization. Unsupported pieces such as attention layers are reported explicitly.

In [1]:
import torch
import torchvision.models as models

from pathnorm import compute_path_norm, compute_path_norms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("torch:", torch.__version__)
print("cuda build:", torch.version.cuda)
print("cudnn:", torch.backends.cudnn.version())


device: cuda
torch: 2.7.1+cu126
cuda build: 12.6
cudnn: 90501


## Example usage

In [2]:
toy_model = torch.nn.Sequential(
    torch.nn.Conv2d(3, 4, kernel_size=3, padding=1, bias=True),
    torch.nn.BatchNorm2d(4),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(kernel_size=2),
    torch.nn.Conv2d(4, 2, kernel_size=3, padding=1, bias=False),
    torch.nn.ReLU(),
    torch.nn.AdaptiveAvgPool2d((1, 1)),
    torch.nn.Flatten(),
    torch.nn.Linear(2, 1),
).to(DEVICE).eval()

print("L1 path-norm:", compute_path_norm(toy_model, input_shape=(3, 32, 32), device=DEVICE))
print("Lq path-norms:", compute_path_norms(toy_model, input_shape=(3, 32, 32), exponents=[1, 2, 4], device=DEVICE))


L1 path-norm: 11.368784422472503
Lq path-norms: [11.368784422472503, 0.31686698682992626, 0.31659415364337984]


## Standard vision model

In [3]:
model = models.resnet18(weights=None).to(DEVICE).eval()
l1 = compute_path_norm(model, input_shape=(3, 224, 224), device=DEVICE)
lqs = compute_path_norms(model, input_shape=(3, 224, 224), exponents=[1, 2, 4], device=DEVICE)

print("L1 path-norm:", l1)
print("Lq path-norms:", lqs)


L1 path-norm: 6.309489099057924e+30
Lq path-norms: [6.309489099057924e+30, 516.1198867529619, 0.16706433037403792]


## Example of an unsupported model

If the computation fails, it points to the exact module or operation that makes the model unsupported.

Here is a tiny example with `GELU`, which falls outside the ReLU path-norm setting since it breaks the existence of rescaling weight symmetries.

In [4]:
unsupported_model = torch.nn.Sequential(
    torch.nn.Conv2d(3, 4, kernel_size=3, padding=1),
    torch.nn.GELU(),
    torch.nn.AdaptiveAvgPool2d((1, 1)),
    torch.nn.Flatten(),
    torch.nn.Linear(4, 2),
)

try:
    compute_path_norm(unsupported_model, input_shape=(3, 32, 32), device=DEVICE)
except ValueError as exc:
    print(exc)


Unsupported path-norm model:
- `1` has unsupported type `GELU`. Currently supported leaves are Conv2d, Linear, ReLU, BatchNorm, MaxPool2d, AvgPool2d, AdaptiveAvgPool2d, Flatten, Identity, and Dropout.
Warnings:
- The model is in training mode. Path-norm is computed with frozen/eval BatchNorm statistics.



For the paper reproductions, use the notebooks in `repro/iclr24` and `repro/icml25`, which build on this same public core.